# 02 · In-memory retrieval baseline

Can a small local embedding model find the reviewed supporting **sections**?
Install `python -m pip install -e ".[dev,ingestion,retrieval]"` in the FishRAG environment.
Run from the repository root or `notebooks/`. Source snapshots must match `data/sources.json`.
This notebook shares `fishrag.ingestion` with notebook 01. 

In [ ]:
import hashlib
import json
import os
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "evals/questions.json").is_file(), "Start in the root or notebooks/."

# Keep model weights, telemetry settings, and generated results local to this workspace.
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from fishrag.ingestion import chunk_documents, load_documents

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
# Official repository revision verified on November 7, 2026; Apache-2.0 model license.
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
ALLOW_MODEL_DOWNLOAD = False  # Set True for the first run; then return to False.
DEVICE = "cpu"
BATCH_SIZE = 16
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
KS = (1, 3, 5, 10)
OUTPUT = ROOT / "outputs/retrieval_baseline"
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(0)
torch.set_num_threads(4)

## 1. Reuse the reviewed corpus and check labels

The shared loader verifies snapshot hashes and titles. The reviewed splitter retains glossary
definitions, list boundaries, parent regulatory qualifiers, section paths, and provenance.
We embed `node.text` only, including any headings/parent context already in that text.
Benchmark facts and answerability labels are never included in the corpus or embedding inputs.
Section identity is the exact pair `(source_id, section)`, not a random chunk ID.

In [ ]:
manifest = json.loads((ROOT / "data/sources.json").read_text(encoding="utf-8"))
benchmark = json.loads((ROOT / "evals/questions.json").read_text(encoding="utf-8"))
questions = benchmark["questions"]
assert len({q["id"] for q in questions}) == len(questions)
for source in manifest["sources"]:
    assert benchmark["source_snapshot_sha256"][source["source_id"]] == source["sha256"]

documents = load_documents(ROOT, manifest)
nodes = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
section_keys = [
    (node.metadata["source_id"], node.metadata["section"]) for node in nodes
]
available_sections = set(section_keys)
parents = {doc.id_: doc for doc in documents}
for node in nodes:
    assert node.metadata == parents[node.ref_doc_id].metadata


def gold_sections(question):
    return {
        (reference["source_id"], heading)
        for reference in question["relevant_sources"]
        for heading in reference["section_headings"]
    }


for question in questions:
    gold = gold_sections(question)
    assert gold <= available_sections, (question["id"], gold - available_sections)
    assert question["answerability"] in {
        "supported",
        "partially_supported",
        "unsupported",
    }
    if question["answerability"] == "unsupported":
        assert not gold and not question["expected_supporting_facts"]
    else:
        assert gold
        assert question["requires_multiple_sections"] == (len(gold) > 1)

print(
    {
        "sections": len(available_sections),
        "chunks": len(nodes),
        "questions": len(questions),
    }
)

## 2. Encode chunks and questions locally

[all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
is a small general-purpose starting point, not a fishing-specific or regulation-aware model.
We pin its repository revision, use CPU float32, no prompt, normalized embeddings, and batch size 16.

The model's token limit differs from the ingestion splitter's tokenizer and 512-token budget.
We **keep the reviewed chunks unchanged** and count inputs that will be truncated. This can
hide supporting facts near a chunk's end even when the section heading matches. Truncation is
a baseline limitation, not an extraction failure. No similarity threshold is used for answerability.

In [ ]:
model = SentenceTransformer(
    MODEL_ID,
    revision=MODEL_REVISION,
    device=DEVICE,
    cache_folder=str(ROOT / ".cache/huggingface/hub"),
    local_files_only=not ALLOW_MODEL_DOWNLOAD,
    trust_remote_code=False,
    model_kwargs={"use_safetensors": True},
)
model.eval()
assert next(model.parameters()).dtype == torch.float32
chunk_texts = [node.text for node in nodes]
question_texts = [question["question"] for question in questions]


def input_lengths(texts):
    return [
        len(
            model.tokenizer(text, truncation=False, add_special_tokens=True)[
                "input_ids"
            ]
        )
        for text in texts
    ]


chunk_lengths = input_lengths(chunk_texts)
question_lengths = input_lengths(question_texts)
truncated_chunks = [
    i for i, length in enumerate(chunk_lengths) if length > model.max_seq_length
]
truncated_questions = [
    i for i, length in enumerate(question_lengths) if length > model.max_seq_length
]
print(
    {
        "model_max_tokens": model.max_seq_length,
        "truncated_chunks": len(truncated_chunks),
        "truncated_questions": len(truncated_questions),
    }
)
for i in truncated_chunks:
    print("Truncated input:", i, chunk_lengths[i], section_keys[i])

chunk_vectors = model.encode(
    chunk_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    precision="float32",
    show_progress_bar=False,
)
question_vectors = model.encode(
    question_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    precision="float32",
    show_progress_bar=False,
)
assert np.isfinite(chunk_vectors).all() and np.isfinite(question_vectors).all()
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1, atol=1e-5)
assert np.allclose(np.linalg.norm(question_vectors, axis=1), 1, atol=1e-5)
scores = question_vectors @ chunk_vectors.T
print("Embedding shapes:", chunk_vectors.shape, question_vectors.shape)

## 3. Rank chunks, then deduplicate sections

Sort all chunks by descending cosine score. Keep the highest-scoring chunk for each unique
source/section pair; ties keep corpus order. `k=5` means five **unique sections**, not five
possibly overlapping chunks. A section's score is its maximum chunk score. 

In [ ]:
def rank_sections(chunk_scores, keys):
    assert len(chunk_scores) == len(keys)
    seen = set()
    ranked = []
    for index in np.argsort(-np.asarray(chunk_scores), kind="stable"):
        key = keys[int(index)]
        if key not in seen:
            seen.add(key)
            ranked.append(
                {
                    "key": key,
                    "chunk_index": int(index),
                    "score": float(chunk_scores[index]),
                }
            )
    return ranked


def metrics_at_k(ranked, gold, k):
    # Undefined for unsupported questions: never create negative relevance labels.
    assert gold and k > 0
    keys = [row["key"] for row in ranked[:k]]
    hits = gold.intersection(keys)
    first_hit = next((rank for rank, key in enumerate(keys, 1) if key in gold), None)
    return {
        "recall": len(hits) / len(gold),
        "reciprocal_rank": 1 / first_hit if first_hit is not None else 0.0,
        "all_required_found": gold.issubset(keys),
    }


# Small hand-checkable fixtures guard against counting overlapping chunks as extra hits.
a, b, c = ("source", "A"), ("source", "B"), ("source", "C")
fixture = rank_sections([0.9, 0.8, 0.7, 0.6], [a, a, b, c])
assert [row["key"] for row in fixture] == [a, b, c]
assert fixture[0]["chunk_index"] == 0
assert metrics_at_k(fixture, {a, b}, 1) == {
    "recall": 0.5,
    "reciprocal_rank": 1.0,
    "all_required_found": False,
}
assert metrics_at_k(fixture, {b}, 1)["reciprocal_rank"] == 0.0
assert metrics_at_k(fixture, {b}, 3)["reciprocal_rank"] == 0.5
assert metrics_at_k(fixture, {a, b}, 3)["all_required_found"]
assert rank_sections([0.5, 0.5], [a, b])[0]["key"] == a

# Identical headings from different sources must not be merged.
assert len(rank_sections([0.9, 0.8], [a, ("other-source", "A")])) == 2
assert metrics_at_k(fixture, {("missing", "section")}, 10)["recall"] == 0.0

rankings = [rank_sections(row, section_keys) for row in scores]
assert all(len(ranked) == len(available_sections) for ranked in rankings)

## 4. Section-level metrics

For a question with gold set G, Recall@k = `|G ∩ top_k_sections| / |G|`.
Reciprocal rank at k is `1 / rank_of_first_gold_section` if that rank is at most k, otherwise zero.
MRR@k is the mean reciprocal rank across a stated question group; it does **not** show whether
all supporting sections were found. For questions requiring multiple sections, we separately
report the proportion with every required section in the top k, plus per-question flags.
Means are macro averages (each question has equal weight).

Evaluate 17 supported questions and the supported portion of the one partially supported
question, with separate groups and an overall 18-question group. The partial label never
becomes fully answerable because the generic section was found. Two unsupported questions
are excluded from relevance metrics and displayed separately. Labels measure annotated section
coverage, not completeness of facts within a truncated chunk, legality, or answer faithfulness.

In [ ]:
per_question = []
for question, ranked in zip(questions, rankings, strict=True):
    gold = gold_sections(question)
    per_question.append(
        {
            "id": question["id"],
            "answerability": question["answerability"],
            "category": question["category"],
            "requires_multiple_sections": question["requires_multiple_sections"],
            "metrics": {str(k): metrics_at_k(ranked, gold, k) for k in KS}
            if gold
            else None,
            "required_section_ranks": [
                {
                    "source_id": key[0],
                    "section": key[1],
                    "rank": next(
                        i for i, row in enumerate(ranked, 1) if row["key"] == key
                    ),
                }
                for key in sorted(gold)
            ],
        }
    )

groups = {
    "supported": [row for row in per_question if row["answerability"] == "supported"],
    "partial_supported_portion": [
        row for row in per_question if row["answerability"] == "partially_supported"
    ],
    "all_labeled": [row for row in per_question if row["metrics"] is not None],
}
summary = []
for name, group in groups.items():
    multi = [row for row in group if row["requires_multiple_sections"]]
    for k in KS:
        row = {
            "group": name,
            "k": k,
            "questions": len(group),
            "recall": float(np.mean([q["metrics"][str(k)]["recall"] for q in group])),
            "mrr": float(
                np.mean([q["metrics"][str(k)]["reciprocal_rank"] for q in group])
            ),
            "multi_section_questions": len(multi),
            "all_required_found_count": sum(
                q["metrics"][str(k)]["all_required_found"] for q in multi
            ),
            "all_required_found_rate": float(
                np.mean([q["metrics"][str(k)]["all_required_found"] for q in multi])
            )
            if multi
            else None,
        }
        summary.append(row)
        print(json.dumps(row))

print("\nPer-question recall and all-required flags:")
for row in per_question:
    if row["metrics"] is not None:
        print(
            row["id"],
            {
                k: (m["recall"], m["all_required_found"])
                for k, m in row["metrics"].items()
            },
        )

## 5. Inspect results and failures

The displays below show the top three unique sections, their best chunk excerpts, and the rank
of **each required section**. Inspect biology, partial support, and all three regulation-exception
questions, then show every labeled question missing a required section at k=10. A general
zone rule can score well while the exception or sanctuary needed for a safe interpretation is
absent. Do not change gold labels just to improve a score. Inspect source metadata and full
sections when the winning chunk omits a qualification.

In [ ]:
by_id = {question["id"]: i for i, question in enumerate(questions)}


def inspect_question(question_id):
    index = by_id[question_id]
    question = questions[index]
    print("\n", question_id, question["question"])
    print(
        "Label:",
        question["answerability"],
        "Missing evidence:",
        question["missing_evidence"],
    )
    print("Required section ranks:", per_question[index]["required_section_ranks"])
    for rank, result in enumerate(rankings[index][:3], 1):
        node = nodes[result["chunk_index"]]
        print(rank, round(result["score"], 4), result["key"])
        print(
            "URL:",
            node.metadata["section_url"],
            "Regulation year:",
            node.metadata["regulation_year"],
        )
        print(
            "Access date:",
            node.metadata["access_date"],
            "Kind:",
            node.metadata["content_kind"],
        )
        print(node.text[:550])


for question_id in (
    "fishrag-001",
    "fishrag-013",
    "fishrag-014",
    "fishrag-015",
    "fishrag-018",
):
    inspect_question(question_id)

failures = [
    row
    for row in per_question
    if row["metrics"] is not None and not row["metrics"]["10"]["all_required_found"]
]
print("\nIncomplete required-section coverage at k=10:")
for row in failures:
    print(row["id"], row["required_section_ranks"])

print("\nUnsupported questions: diagnostic similarity only; no relevance metrics.")
for question in questions:
    if question["answerability"] == "unsupported":
        inspect_question(question["id"])

print("\nTruncation inspection: encoded tail versus full source tail")
for source_id, heading in (
    ("ontario-zone-17", "Waterbody exceptions"),
    (
        "ontario-learn-to-fish",
        (
            "Identifying, catching and handling fish species > "
            "Smallmouth and largemouth bass"
        ),
    ),
):
    index = next(i for i, key in enumerate(section_keys) if key == (source_id, heading))
    encoded = model.tokenizer(
        chunk_texts[index], truncation=True, max_length=model.max_seq_length
    )["input_ids"]
    print(source_id, heading)
    print("Encoded tail:", model.tokenizer.decode(encoded)[-400:])
    print("Full chunk tail:", chunk_texts[index][-400:])

## Review of the first local run

With the pinned model and reviewed corpus, all 18 labeled queries had mean Recall@10
0.9352 and MRR@10 0.9074. Only 6 of 9 multi-section queries found all required sections.
These are development-benchmark observations, not answer-generation results.

- `fishrag-001`: the bass identification section ranked first.
- `fishrag-013`: the zone-wide walleye rule ranked first, but **Waterbody exceptions**
  ranked tenth. A top-three result would miss the annotated Scugog closure section.
- `fishrag-015`: the zone-wide walleye rule and the species exception ranked first and
  second, so both annotated sections were available at k=3.
- `fishrag-014`: the Chemong sanctuary section ranked first and the zone-wide bass rule
  second, but the general **Fish sanctuaries** definition ranked 36th. This counts as
  incomplete coverage under the existing labels even though the local closure was found.
- `fishrag-009` missed the aggregate-limit definition at k=10 (rank 17), and
  `fishrag-012` missed the livewell definition (rank 102). A good first hit did not
  retrieve every required definition.
- Eighteen chunk inputs exceeded the model's 256-token limit; no query did. The bass
  section's encoded tail cuts off before the crayfish preference, and the waterbody
  exceptions input cuts off before the final Trent River walleye season. The full local
  source text is preserved, but not all of it contributes to these embeddings.
- The supported portion of `fishrag-018` ranked first. That does not establish an exact
  Rice Lake depth. The unsupported lure-colour question still had a top cosine score
  around 0.542, illustrating why a similarity score cannot establish answerability.

Scores may change with package/runtime changes, source/model revisions and settings are saved below.


## 6. Save an auditable local run

The ignored output directory stores vectors, corpus text and provenance, settings, rankings,
and metric rows. These are experiment artifacts, not a vector database. Settings record input
hashes so later changes to extraction, snapshots, or labels are visible. Re-running overwrites
this local baseline directory.

In [ ]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


settings = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_license": "Apache-2.0",
    "model_max_seq_length": model.max_seq_length,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "torch_threads": torch.get_num_threads(),
    "embedding_precision": "float32",
    "embedding_dimension": chunk_vectors.shape[1],
    "normalized_embeddings": True,
    "prompt": None,
    "embedding_input": "node.text / question text only",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "chunk_tokenizer": "LlamaIndex default (cl100k_base)",
    "section_score": "maximum chunk cosine",
    "tie_break": "corpus order",
    "ks": list(KS),
    "truncated_chunk_indices": truncated_chunks,
    "truncated_question_indices": truncated_questions,
    "source_snapshot_sha256": benchmark["source_snapshot_sha256"],
    "input_file_sha256": {
        name: file_hash(ROOT / name)
        for name in (
            "data/sources.json",
            "evals/questions.json",
            "src/fishrag/ingestion.py",
            "notebooks/02_retrieval_baseline.ipynb",
        )
    },
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in (
            "sentence-transformers",
            "torch",
            "transformers",
            "numpy",
            "llama-index-core",
            "beautifulsoup4",
            "huggingface-hub",
        )
    },
}
report = {"settings": settings, "summary": summary, "per_question": per_question}
(OUTPUT / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(OUTPUT / "rankings.json").write_text(
    json.dumps(
        {
            question["id"]: [
                {
                    "source_id": row["key"][0],
                    "section": row["key"][1],
                    "chunk_index": row["chunk_index"],
                    "score": row["score"],
                }
                for row in ranked
            ]
            for question, ranked in zip(questions, rankings, strict=True)
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUTPUT / "corpus.json").write_text(
    json.dumps(
        [
            {"chunk_index": index, "text": node.text, "metadata": node.metadata}
            for index, node in enumerate(nodes)
        ],
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
np.savez(OUTPUT / "embeddings.npz", chunks=chunk_vectors, questions=question_vectors)
print("Local run saved to", OUTPUT)

## Interpretation limits

This is a small, manually labelled development benchmark, not an independent test set. Weights
and settings are fixed before looking at these results. Broad section labels can award a hit
even when the winning chunk lacks an expected fact; overlapping parent context can also make
an unannotated section useful without earning label credit. Inspect failures rather than
silently broadening relevance labels. MRR can be high when one necessary exception is still
missing, which is why all-required-section coverage is reported separately.

Generic fishing similarity does not establish current Rice Lake conditions or answerability.
The partial question is scored only for its labeled general guidance. Unsupported questions
receive diagnostic rankings but no fabricated relevance judgments or metric denominators.
No generated answers or answer-faithfulness results are produced.